In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/FYP_SR_Data")

for folder in ["DIV2K", "Set5", "Set14", "BSD100", "Urban100"]:
    (DATA_ROOT / folder).mkdir(parents=True, exist_ok=True)

print(DATA_ROOT)

/content/drive/MyDrive/FYP_SR_Data


In [27]:
datasets = ["Set5", "Set14", "BSD100", "Urban100"]
scales = ["x2", "x3", "x4"]

data_root = Path("/content/drive/MyDrive/FYP_SR_Data")
metrics_dir = data_root / "results" / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)

In [29]:
import numpy as np
import pandas as pd

from pathlib import Path
from time import perf_counter
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

def evaluate_dataset(dataset_name, scale):
    dataset_root = data_root / dataset_name

    hr_dir = dataset_root / f"{dataset_name}_HR"
    lr_dir = dataset_root / f"{dataset_name}_LR_{scale}"

    records = []

    for hr_path in sorted(hr_dir.glob("*")):
        if hr_path.suffix.lower() not in [".png", ".jpg", ".jpeg"]:
            continue

        lr_path = lr_dir / hr_path.name

        if not lr_path.exists():
            raise FileNotFoundError(f"Missing LR file: {lr_path}")

        hr_image = Image.open(hr_path).convert("RGB")
        lr_image = Image.open(lr_path).convert("RGB")

        start = perf_counter()

        sr_image = lr_image.resize(
            hr_image.size,
            Image.Resampling.BICUBIC
        )

        runtime_ms = (perf_counter() - start) * 1000

        hr_array = np.array(hr_image)
        sr_array = np.array(sr_image)

        records.append({
            "dataset": dataset_name,
            "image": hr_path.name,
            "scale": scale,
            "method": "bicubic",
            "psnr": peak_signal_noise_ratio(
                hr_array,
                sr_array,
                data_range=255
            ),
            "ssim": structural_similarity(
                hr_array,
                sr_array,
                channel_axis=2,
                data_range=255
            ),
            "runtime_ms": runtime_ms
        })

    return pd.DataFrame(records)

In [31]:
all_results = []

for dataset_name in datasets:
    for scale in scales:
        print(f"Running {dataset_name} at {scale}")

        result = evaluate_dataset(dataset_name, scale)
        all_results.append(result)

        filename = f"{dataset_name}_{scale}_bicubic.csv"
        result.to_csv(metrics_dir / filename, index=False)

        print(
            f"Average PSNR: {result['psnr'].mean():.4f} dB | "
            f"Average SSIM: {result['ssim'].mean():.4f}"
        )

all_results_df = pd.concat(all_results, ignore_index=True)

Running Set5 at x2
Average PSNR: 31.7868 dB | Average SSIM: 0.9178
Running Set5 at x3
Average PSNR: 28.6067 dB | Average SSIM: 0.8529
Running Set5 at x4
Average PSNR: 26.6902 dB | Average SSIM: 0.7899
Running Set14 at x2
Average PSNR: 28.3194 dB | Average SSIM: 0.8566
Running Set14 at x3
Average PSNR: 25.7172 dB | Average SSIM: 0.7613
Running Set14 at x4
Average PSNR: 24.2384 dB | Average SSIM: 0.6854
Running BSD100 at x2
Average PSNR: 28.2729 dB | Average SSIM: 0.8459
Running BSD100 at x3
Average PSNR: 25.8813 dB | Average SSIM: 0.7388
Running BSD100 at x4
Average PSNR: 24.6403 dB | Average SSIM: 0.6608
Running Urban100 at x2
Average PSNR: 25.4295 dB | Average SSIM: 0.8378
Running Urban100 at x3
Average PSNR: 23.0072 dB | Average SSIM: 0.7329
Running Urban100 at x4
Average PSNR: 21.6991 dB | Average SSIM: 0.6517


In [32]:
summary = (
    all_results_df
    .groupby(["dataset", "scale", "method"])
    [["psnr", "ssim", "runtime_ms"]]
    .mean()
    .reset_index()
)

summary.to_csv(
    metrics_dir / "bicubic_summary_all_datasets.csv",
    index=False
)

summary

,dataset,scale,method,psnr,ssim,runtime_ms
0,BSD100,x2,bicubic,28.272943,0.845896,4.149980
1,BSD100,x3,bicubic,25.881261,0.738794,3.053478
2,BSD100,x4,bicubic,24.640267,0.660768,3.716162
3,Set14,x2,bicubic,28.319449,0.856582,4.808746
4,Set14,x3,bicubic,25.717207,0.761250,4.081062
5,Set14,x4,bicubic,24.238399,0.685370,3.629712
6,Set5,x2,bicubic,31.786760,0.917804,2.366635
7,Set5,x3,bicubic,28.606664,0.852941,2.187460
8,Set5,x4,bicubic,26.690237,0.789909,1.799780
9,Urban100,x2,bicubic,25.429463,0.837845,17.571724
